In [1]:
%load_ext autoreload
%autoreload 2

# --- Standard libs ---
import numpy as np
import pandas as pd
import time
import sys
import serial
from pathlib import Path


PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

# --- Project imports ---
from Random.fakes import (FakeDIM, FakeRunze, FakePump, FakeGilson, FakeSyrPump)
from Ecosystems.reaction_segment_generation import RSG, RSGState
from Ecosystems.segmentation import (SegmentationState, Segmentation)

In [2]:
# --- Instantiate fake instruments ---
fake_dim = FakeDIM()
fake_runze = FakeRunze()
fake_pump = FakePump()
fake_gilson = FakeGilson()
fake_syrpump = FakeSyrPump()

# --- Create ecosystems ---
rsg = RSG(gilson = fake_gilson, pump = fake_syrpump)
seg = Segmentation(dim=fake_dim, runze=fake_runze, carrier_pump=fake_pump, rsg=rsg)


In [4]:
print(seg.state)

SegmentationState.IDLE


In [4]:
# -------------------------------
# Test Notebook for Fakes & Guards
# -------------------------------

# 1️⃣ Check initial states
print("Initial states:")
print("RSG state:", rsg.state)
print("Segmentation state:", seg.state)
print("DIM position:", fake_dim.read_pos())
print("-" * 40)

# 2️⃣ Test DIM assert methods
print("Testing DIM asserts:")

# Force LOAD
fake_dim.load()
try:
    fake_dim.assert_load()
    print("✅ assert_load passed")
except RuntimeError as e:
    print("❌ assert_load failed:", e)

# Force INJECT
fake_dim.inject()
try:
    fake_dim.assert_inject()
    print("✅ assert_inject passed")
except RuntimeError as e:
    print("❌ assert_inject failed:", e)

# Invalid state simulation
fake_dim.position = "X"
try:
    fake_dim.assert_load()
except RuntimeError as e:
    print("✅ caught invalid load:", e)
try:
    fake_dim.assert_inject()
except RuntimeError as e:
    print("✅ caught invalid inject:", e)
print("-" * 40)

# 3️⃣ Test RSG _require_idle
print("Testing RSG _require_idle:")

# Should pass
try:
    rsg._require_idle()
    print("✅ _require_idle passed (IDLE)")
except RuntimeError as e:
    print("❌ _require_idle failed:", e)

# Force RSG to RUNNING
rsg.state = RSGState.RUNNING
try:
    rsg._require_idle()
except RuntimeError as e:
    print("✅ caught _require_idle while RUNNING:", e)

# Reset
rsg.state = RSGState.IDLE
print("-" * 40)

# 4️⃣ Test RSG primitives with fakes
print("Testing RSG primitives with fakes:")

# Pickup from vial
rsg.pickup_from_vial("rack1", 1, volume=10.0, rate=0.05)

# Dispense into vial
rsg.dispense_in_vial("rack1", 2, volume=15.0, rate=0.5)

# Take air gap
rsg.take_air_gap(volume=5.0, rate=0.05)

# Prepickup
rsg.prepickup(volume=10.0, rate=0.05)

# Dispense into DIM
rsg.dispense_in_dim(volume=20.0, rate=0.5)

# Dispense into waste
rsg.dispense_in_waste(volume=25.0, rate=0.5)
print("-" * 40)

# 5️⃣ Test build_reaction
print("Testing build_reaction:")
reaction_plan = [
    {"module": "rack1", "vial": 1, "volume": 5},
    {"module": "rack1", "vial": 2, "volume": 7},
]
meta = rsg.build_reaction(reaction_plan, air_gap_between=1.0)
print("Reaction metadata:", meta)
print("-" * 40)

# 6️⃣ Test wash_sequence
print("Testing wash_sequence:")
rsg.wash_sequence(solvent_volume=20.0, air_gap=2.0)
print("-" * 40)

print("✅ All fake tests completed")


Initial states:
RSG state: RSGState.RUNNING
Segmentation state: SegmentationState.IDLE
DIM position: X
----------------------------------------
Testing DIM asserts:
[FAKE DIM] Switched to LOAD
[FAKE DIM] assert_load passed
✅ assert_load passed
[FAKE DIM] Switched to INJECT
[FAKE DIM] assert_inject passed
✅ assert_inject passed
✅ caught invalid load: DIM must be in LOAD. Current position: X
✅ caught invalid inject: DIM must be in INJECT. Current position: X
----------------------------------------
Testing RSG _require_idle:
❌ _require_idle failed: RSG is busy or in error state: RSGState.RUNNING
✅ caught _require_idle while RUNNING: RSG is busy or in error state: RSGState.RUNNING
----------------------------------------
Testing RSG primitives with fakes:
[Gilson] go_into_vial -> rack1, 1
[Pump] prepare -> rate=0.05, volume=10.0, dir=WDR
[Pump] start
[Pump] stop
[Gilson] go_into_vial -> rack1, 2
[Pump] prepare -> rate=0.5, volume=15.0, dir=INF
[Pump] start
[Pump] stop
[Gilson] ensure_z_sa

In [4]:
# Test Segmentation _require_idle
print("Testing Segmentation _require_idle:")
try:
    seg._require_idle()
    print("✅ Segmentation _require_idle passed (IDLE)")
except RuntimeError as e:
    print("❌ Segmentation _require_idle failed:", e)

# Force it to FLOWING
seg.state = SegmentationState.FLOWING
try:
    seg._require_idle()
except RuntimeError as e:
    print("✅ caught _require_idle while FLOWING:", e)

# Reset
seg.state = SegmentationState.IDLE
print("-" * 40)


Testing Segmentation _require_idle:
✅ Segmentation _require_idle passed (IDLE)
✅ caught _require_idle while FLOWING: Segmentation is busy or in error state: SegmentationState.FLOWING
----------------------------------------


In [3]:
print("Testing multiple slug launches:")

for i in range(2):
    print(f"--- Launching slug {i+1} ---")
    fake_dim.load()  # reset DIM to LOAD
    seg.state = SegmentationState.LOOP_LOADED  # ensure precondition
    seg.launch_segment(flowrate_ul_min=100 + i*20)  # vary the flow rate




Testing multiple slug launches:
--- Launching slug 1 ---
[FAKE DIM] Switched to LOAD
[FAKE DIM] Switched to INJECT
[FAKE RUNZE] Moved to position 2
[FAKE PUMP] Flow rate set to 100
[FAKE PUMP] Flow started
--- Launching slug 2 ---
[FAKE DIM] Switched to LOAD
[FAKE DIM] Switched to INJECT
[FAKE RUNZE] Already at position 2
[FAKE PUMP] Flow rate set to 120
[FAKE PUMP] Flow started


In [4]:
print("Testing RSG -> Segmentation integration:")

reaction_plan = [
    {"module": "rack1", "vial": 1, "volume": 5},
    {"module": "rack1", "vial": 2, "volume": 7},
]

meta = rsg.build_reaction(reaction_plan, air_gap_between=1.0)
print("RSG built reaction metadata:", meta)

# Then launch it via Segmentation
fake_dim.load()
seg.launch_segment(flowrate_ul_min=0.5)


Testing RSG -> Segmentation integration:
[Gilson] go_into_vial -> rack1, 1
[Pump] prepare -> rate=0.05, volume=5, dir=WDR
[Pump] start
[Pump] stop
[Gilson] ensure_z_safe
[Pump] prepare -> rate=0.05, volume=1.0, dir=WDR
[Pump] start
[Pump] stop
[Gilson] go_into_vial -> rack1, 2
[Pump] prepare -> rate=0.05, volume=7, dir=WDR
[Pump] start
[Pump] stop
RSG built reaction metadata: {'total_volume_ul': 13.0, 'num_components': 2}
[FAKE DIM] Switched to LOAD


RuntimeError: Invalid state transition: required SegmentationState.LOOP_LOADED, but current state is SegmentationState.FLOWING